# 06 · Personalized Search (TF-IDF + LightGBM)

**Amazon Reviews 2023 — Electronics** · Real-time recommendation system

This bonus adds a **personalized search** flow, a different shape from browse-based recommendation:

1. **Text retrieval** — the user types a query; we match it against **item titles** with **TF-IDF** cosine similarity and take the top candidates. This guarantees the results are *relevant to the query*.
2. **Personalized re-ranking** — we then re-order those candidates with our trained **LightGBM** ranker, using **user features** (category affinity, the two-tower preference score, item popularity, …). This makes the *same query* return a *different order for different users* — personalized search results.

So text handles *relevance* (candidate generation) and the ranker handles *personalization* (ordering). We reuse the exact ranker and feature code from notebook `05`.

## 0 · Setup

In [1]:
import sys
from pathlib import Path
try:
    import google.colab  # noqa
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/rec-system')  # <-- adjust to your Drive path
except ImportError:
    PROJECT = Path('..').resolve()
DATA_DIR, ART_DIR = PROJECT / 'data', PROJECT / 'artifacts'
sys.path.insert(0, str(PROJECT / 'src'))

In [2]:
import numpy as np, pandas as pd, torch, lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from recsys.models.two_tower import TwoTower
from recsys import features as F
pd.set_option('display.max_colwidth', 70)
device = torch.device('cpu')

## 1 · Load data, the two-tower retriever, and the LightGBM ranker

We need the item embeddings (for the personalization `retrieval_score` feature) and the trained ranker + feature tables from notebook `05`.

In [3]:
train = pd.read_parquet(DATA_DIR / 'splits' / 'train.parquet')
test  = pd.read_parquet(DATA_DIR / 'splits' / 'test.parquet')
items = pd.read_parquet(DATA_DIR / 'items.parquet').sort_values('item_idx').reset_index(drop=True)
n_users = int(pd.read_parquet(DATA_DIR / 'user_map.parquet').shape[0])
n_items = int(pd.read_parquet(DATA_DIR / 'item_map.parquet').shape[0])
train['positive'] = train['rating'] >= 4
split_ts = int(train['timestamp'].max())

cat_of_item = np.load(ART_DIR / 'item_categories.npy')
n_cats = int(cat_of_item.max()) + 1
item_emb = np.load(ART_DIR / 'two_tower_item_emb.npy').astype('float32')   # L2-normalized

model = TwoTower(n_users, n_items, n_cats, emb=64, hidden=(128,), out_dim=64, temperature=0.1)
model.load_state_dict(torch.load(ART_DIR / 'two_tower.pt', map_location='cpu'))
model.set_item_categories(cat_of_item); model.eval()
ranker = lgb.Booster(model_file=str(ART_DIR / 'lgbm_ranker.txt'))

user_stats = F.compute_user_stats(train, cat_of_item, split_ts)
item_stats = F.compute_item_stats(train, items[['item_idx','average_rating','rating_number','price']], n_items, split_ts)
user_cat   = F.compute_user_cat_counts(train, cat_of_item)
print('loaded:', n_items, 'items,', n_users, 'users')

loaded: 20450 items, 25546 users


## 2 · Build the TF-IDF index over item titles

We vectorize every item **title** with TF-IDF (unigrams + bigrams, English stop-words removed). A search query is transformed into the same space and scored by cosine similarity against all items.

In [4]:
titles = items['title'].fillna('').tolist()   # aligned to item_idx 0..n_items-1
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=50000)
tfidf = vectorizer.fit_transform(titles)       # [n_items, vocab]
print('TF-IDF matrix:', tfidf.shape, '| vocabulary:', len(vectorizer.vocabulary_))

TF-IDF matrix: (20450, 50000) | vocabulary: 50000


In [5]:
def text_search(query, top_n=50):
    """Return (item_idxs, text_scores) for the top-N title matches to `query`."""
    q = vectorizer.transform([query])
    scores = np.asarray((tfidf @ q.T).todense()).ravel()
    idx = np.argsort(-scores)[:top_n]
    idx = idx[scores[idx] > 0]                  # keep only actual matches
    return idx, scores[idx]

# sanity check: a plain (non-personalized) text search
idx, sc = text_search('wireless earbuds', top_n=5)
items.set_index('item_idx').loc[idx, ['title', 'category']]

,title,category
item_idx,,
9401,FAMOO Wireless Earbuds,Open-Ear Headphones
14765,yobola Wireless Earbuds - Pink,Earbud Headphones
12189,Outdoor Tech Ravens Wireless Earbuds - True Wireless Earbuds - Spo...,Earbud Headphones
3503,GOLREX Wireless Earbuds,Earbud Headphones
14136,"Pluto True Wireless Earbuds, Sleek Earbuds Wireless Bluetooth 5.3 ...",Earbud Headphones


## 3 · Personalized re-ranking with user features

Given a query **and** a user, we take the TF-IDF candidates and re-rank them with LightGBM. The `retrieval_score` feature becomes the **two-tower cosine** between this user and each candidate item (i.e. *would this user like this item?*), and the other features add category affinity, popularity, recency, etc. — so the ordering personalizes to the user.

In [6]:
def personalized_search(query, user_idx, k=8, top_n=50):
    cand, tscore = text_search(query, top_n)
    if len(cand) == 0:
        return None
    with torch.no_grad():
        uvec = model.user_vectors(torch.tensor([user_idx]))[0]      # normalized [64]
    rscore = item_emb[cand] @ uvec                                  # two-tower cosine per candidate
    df = pd.DataFrame({'user_idx': user_idx, 'item_idx': cand, 'retrieval_score': rscore})
    feat = F.build_features(df, user_stats, item_stats, user_cat, cat_of_item)
    feat['lgbm'] = ranker.predict(feat[F.FEATURE_COLS])
    feat['text_score'] = tscore
    return feat.sort_values('lgbm', ascending=False).head(k)

def as_table(df):
    return items.set_index('item_idx').loc[df['item_idx'], ['title', 'category']].reset_index(drop=True)

## 4 · Demo - the same query, two different users

We pick two users whose histories are dominated by **different categories**, then run the *same* query for both. Text retrieval gives the same relevant candidate pool; the personalized re-rank orders them differently for each user.

In [7]:
# find two users with contrasting top categories (from their training history)
tr = train.merge(items[['item_idx', 'category']], on='item_idx')
top_cat = tr.groupby('user_idx')['category'].agg(lambda s: s.value_counts().index[0])
seen_ct = train.groupby('user_idx').size()
from collections import defaultdict
by_cat = defaultdict(list)
for u in test.user_idx.unique():
    if seen_ct.get(u, 0) >= 15 and u in top_cat.index:
        by_cat[top_cat[u]].append(int(u))
cats_sorted = sorted(by_cat, key=lambda c: -len(by_cat[c]))
userA, catA = by_cat[cats_sorted[0]][0], cats_sorted[0]
userB, catB = by_cat[cats_sorted[1]][0], cats_sorted[1]
print(f'User A = #{userA} (mostly {catA!r})')
print(f'User B = #{userB} (mostly {catB!r})')

User A = #20003 (mostly 'Earbud Headphones')
User B = #4488 (mostly 'USB Cables')


In [8]:
QUERY = 'charger'   # a broad query whose matches span many categories
print(f'QUERY: {QUERY!r}\n')

cand, _ = text_search(QUERY, top_n=8)
print('--- Non-personalized (TF-IDF text relevance only) ---')
print(items.set_index('item_idx').loc[cand, ['title', 'category']].reset_index(drop=True).to_string())

QUERY: 'charger'

--- Non-personalized (TF-IDF text relevance only) ---
                                                                                                                                                                                                    title             category
0                                                                MacBook Pro Charger for MacBook Air Charger 96W MacBook Charger for Mac Charger USB C Laptop Charger, Ipad Charger Included Type C Cable  Chargers & Adapters
1                                                         OXSANK Multi Charger Cable Type C Charger Fast Charging 3 in 1 Charging Cable iPhone Charger, Automatic Storage Anti-Winding Charger Cord (Red)     Lightning Cables
2     Mac Book Pro Charger, ShuOne 65W USB C Charger Power Adapter, Portable Laptop Charger with 4k HDMI, USB 2.0, USB-C, Ethernet Port, Fast Charger with 6Ft Extension Cord Suitable for MacBook&Switch  Chargers & Adapters
3  Chromebook Charger, 45W 65W Type 

In [9]:
print(f'--- Personalized for User A  (#{userA}, {catA}) ---')
print(as_table(personalized_search(QUERY, userA)).to_string())

--- Personalized for User A  (#20003, Earbud Headphones) ---
                                                                                                                                                                                                     title           category
0                                                          OXSANK Multi Charger Cable Type C Charger Fast Charging 3 in 1 Charging Cable iPhone Charger, Automatic Storage Anti-Winding Charger Cord (Red)   Lightning Cables
1                   5Pack[Apple MFi Certified] USB C to Lightning Cable 10/10/6/6/6 FT PD 20W iPhone Fast Charger Cable for USB C Charger,Lighting to Type C Charger Code for iPhone 13/12/11/iPad/AirPods   Lightning Cables
2          Pluggify USB C Cable 2 Pack Type C Charger Fast Charging: USB C Charger [6.6FT+6.6FT] Durable Nylon Braided Android Samsung Charger with Galaxy S10 S9 S8, Note 10 9 LG, PS5 Controller, Switch         USB Cables
3  iPad Charger iPhone Charger [Apple MFi Certified

In [10]:
print(f'--- Personalized for User B  (#{userB}, {catB}) ---')
print(as_table(personalized_search(QUERY, userB)).to_string())

--- Personalized for User B  (#4488, USB Cables) ---


                                                                                                                                                                                                  title          category
0                             18650 li-ion Battery Charger, Suitable for 3.7v Battery 20700 10440 14500 18500 16340 17500 18650 Charger Charger, USB Single Slot Battery Charger (Battery not Included)  Battery Chargers
1       Pluggify USB C Cable 2 Pack Type C Charger Fast Charging: USB C Charger [6.6FT+6.6FT] Durable Nylon Braided Android Samsung Charger with Galaxy S10 S9 S8, Note 10 9 LG, PS5 Controller, Switch        USB Cables
2                                                                                                                                                                         Forlleco Dual Battery Charger  Battery Chargers
3            TALK WORKS USB-C Wall Charger for MacBook Laptop - 30W Fast Wall Charger Block Cube with Charging Cable & Foldable 

In [11]:
# quantify the personalization: how much do the two users' ranked lists differ?
la = personalized_search(QUERY, userA, k=8)['item_idx'].tolist()
lb = personalized_search(QUERY, userB, k=8)['item_idx'].tolist()
same_pos = sum(1 for x, y in zip(la, lb) if x == y)
shared   = len(set(la) & set(lb))
print(f'Top-8 for the two users: {same_pos}/8 items in the SAME position, '
      f'{shared}/8 items shared overall.')
print('-> same query, relevant to both, but re-ordered per user = personalized search.')

Top-8 for the two users: 0/8 items in the SAME position, 3/8 items shared overall.


-> same query, relevant to both, but re-ordered per user = personalized search.


> **Interpretation.** All three lists are **relevant** to the query (TF-IDF guarantees the titles match). But the **order differs by user**: the LightGBM re-rank promotes the candidates each user is most likely to prefer, driven by their two-tower `retrieval_score` and category affinity. This is exactly *personalized search* — the same words, ranked for *you*. A non-personalized engine would show every user the middle list (pure text relevance).

## 5 · Export the search index (optional, for the webapp)

In [12]:
import pickle, scipy.sparse
with open(ART_DIR / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
scipy.sparse.save_npz(ART_DIR / 'tfidf_matrix.npz', tfidf)
print('saved tfidf_vectorizer.pkl and tfidf_matrix.npz ->', ART_DIR)

saved tfidf_vectorizer.pkl and tfidf_matrix.npz -> F:\Rec_System\artifacts


---


**Produced:** a TF-IDF title index + a `personalized_search(query, user)` function that retrieves by text relevance and re-ranks with the LightGBM model using user features. The demo shows the same query producing different, personalized orderings for two different users. The index is exported so the webapp could add a search bar.